In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D11 — Siyaram Silk Mills Limited
#       Investor Presentation Q4 & FY24
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip install -q pymupdf pymupdf4llm pandas

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import re

import fitz
import pandas as pd
import pymupdf4llm

DOCUMENT_ID = "D11"
DOCUMENT_NAME = "Siyaram Silk Mills Limited — Investor Presentation | Q4 & FY24"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "pymupdf4llm page-aware Markdown conversion with "
    "native-token coverage checking and deterministic "
    "PyMuPDF layout-aware fallback"
)

LLM_INPUT_REPRESENTATION = "Structural Markdown"

SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952"
EXPECTED_PAGE_COUNT = 10

EXPECTED_RECORD_COUNT = 199

EXPECTED_CATEGORY_COUNTS = {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Source Location"
]

VALUE_ALLOWED_TYPES = (str, int, float, type(None))
ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

EXCLUDED_DIVIDER_PAGES = {3, 7}
EXPECTED_ACTIVE_PAGES = {1, 2, 4, 5, 6, 8, 9, 10}

EXPECTED_QUALIFIED_VALUES = {
    "800+",
    "~100Mn",
    "245+",
    "~1.85L",
    "~4.5Mn",
    "5Mn and counting…"
}

MIN_NATIVE_TOKEN_COVERAGE = 0.80

OUTPUT_DIR = Path("outputs_D11_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DIAGNOSTICS_PATH = OUTPUT_DIR / "D11_branch_B_source_diagnostics.json"
PAGE_DIAGNOSTICS_PATH = OUTPUT_DIR / "D11_branch_B_page_diagnostics.csv"
CONVERSION_PAGE_AUDIT_PATH = OUTPUT_DIR / "D11_branch_B_conversion_page_audit.csv"
STRUCTURAL_MARKDOWN_PATH = OUTPUT_DIR / "D11_branch_B_structural_markdown.md"
CONVERSION_INTEGRITY_PATH = OUTPUT_DIR / "D11_branch_B_conversion_integrity.json"
REPRESENTATION_PATH = OUTPUT_DIR / "D11_branch_B_representation.json"
PROMPT_PATH = OUTPUT_DIR / "D11_branch_B_prompt.txt"
EXPERIMENT_METADATA_PRE_PATH = OUTPUT_DIR / "D11_branch_B_experiment_metadata_pre.json"
RAW_RESPONSE_PATH = OUTPUT_DIR / "D11_branch_B_raw_response.txt"
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D11_branch_B_parsed_extraction.json"
TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR / "D11_branch_B_technical_diagnostics.json"
)
EXPERIMENT_METADATA_PATH = OUTPUT_DIR / "D11_branch_B_experiment_metadata.json"
EXPERIMENT_SUMMARY_PATH = OUTPUT_DIR / "D11_branch_B_experiment_summary.json"

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Frozen Stage 1 records:", EXPECTED_RECORD_COUNT)



In [ ]:
# ============================================================
# 1. Upload and verify the exact original PDF
# ============================================================

uploaded = files.upload()

pdf_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]

if len(pdf_paths) != 1:
    raise ValueError("Upload exactly one original D11 PDF.")

SOURCE_PATH = pdf_paths[0]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D11 source format.")

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D11 PDF does not match the frozen Stage 1 source identity."
    )

pdf_document = fitz.open(SOURCE_PATH)
PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = PAGE_COUNT == EXPECTED_PAGE_COUNT

if not PAGE_COUNT_VALID:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {PAGE_COUNT}."
    )

page_rows = []
page_native_text = {}

for page_number, page in enumerate(pdf_document, start=1):

    native_text = page.get_text("text") or ""
    page_native_text[page_number] = native_text

    page_rows.append({
        "Page Number": page_number,
        "Native Character Count": len(native_text),
        "Native Word Count": len(native_text.split()),
        "Non-empty Line Count": len([
            line for line in native_text.splitlines()
            if line.strip()
        ]),
        "Text Block Count": len([
            block for block in page.get_text("blocks")
            if str(block[4]).strip()
        ]),
        "Drawing Count": len(page.get_drawings()),
        "Embedded Image Count": len(page.get_images(full=True)),
        "Width": float(page.rect.width),
        "Height": float(page.rect.height),
        "Orientation": (
            "landscape"
            if page.rect.width > page.rect.height
            else "portrait"
        ),
        "PDF Rotation": int(page.rotation)
    })

page_diagnostics_df = pd.DataFrame(page_rows)

FULL_NATIVE_TEXT = "\n".join(
    page_native_text[page_number]
    for page_number in range(1, PAGE_COUNT + 1)
)

TOTAL_NATIVE_CHARACTERS = len(FULL_NATIVE_TEXT)
TEXT_EXTRACTABLE = bool(FULL_NATIVE_TEXT.strip())
OCR_REQUIRED = not TEXT_EXTRACTABLE

if OCR_REQUIRED:
    raise ValueError(
        "D11 is expected to contain a usable native text layer. "
        "OCR is not part of Branch B for this document."
    )

SOURCE_MARKER_PATTERNS = {
    "presentation_title":
        r"Investor Presentation\s*\|\s*Q4\s*&\s*FY24",
    "safe_harbor":
        r"Safe Harbor",
    "management_commentary":
        r"Management Commentary",
    "quarterly_business_performance":
        r"Quarterly Business Performance",
    "profit_and_loss_statement":
        r"Q4FY24 Profit\s*&\s*Loss Statement",
    "company_profile":
        r"Our Legacy,\s*Our Future",
    "corporate_timeline":
        r"We Improve\.\s*Grow\.\s*Accelerate",
    "operational_footprint":
        r"We serve multiple end markets"
}

SOURCE_MARKER_STATUS = {
    name: bool(
        re.search(pattern, FULL_NATIVE_TEXT, flags=re.IGNORECASE)
    )
    for name, pattern in SOURCE_MARKER_PATTERNS.items()
}

ALL_SOURCE_MARKERS_PRESENT = all(SOURCE_MARKER_STATUS.values())

SOURCE_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "native_text_character_count": TOTAL_NATIVE_CHARACTERS,
    "native_text_word_count": len(FULL_NATIVE_TEXT.split()),
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "source_marker_status": SOURCE_MARKER_STATUS,
    "all_source_markers_present": ALL_SOURCE_MARKERS_PRESENT
}

SOURCE_DIAGNOSTICS_PATH.write_text(
    json.dumps(SOURCE_DIAGNOSTICS, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(SOURCE_DIAGNOSTICS, ensure_ascii=False, indent=2))
display(page_diagnostics_df)

if not ALL_SOURCE_MARKERS_PRESENT:
    raise ValueError(
        "One or more expected D11 source regions are not recoverable "
        "from the original PDF text layer."
    )


In [ ]:
# ============================================================
# 2. Conversion of every page to one non-duplicated structural representation
# ============================================================

def source_tokens(text):
    """
    Tokenisation for source-coverage diagnostics only.
    No transformed copy is supplied to the model.
    """
    return {
        token.casefold()
        for token in re.findall(
            r"[A-Za-zÀ-ÿ0-9][A-Za-zÀ-ÿ0-9.,+%&'’/-]*",
            str(text or "")
        )
        if len(token) >= 2
    }


def native_text_coverage(converted_text, native_text):
    native = source_tokens(native_text)
    converted = source_tokens(converted_text)

    if not native:
        return 1.0

    return len(native & converted) / len(native)


def layout_block_fallback(page):
    """
    Deterministic layout-aware fallback.

    Native block text is retained without semantic rewriting.
    Bounding-box coordinates are added only as structural metadata.
    """
    blocks = [
        block
        for block in page.get_text("blocks")
        if str(block[4]).strip()
    ]

    blocks = sorted(
        blocks,
        key=lambda block: (
            round(float(block[1]), 2),
            round(float(block[0]), 2)
        )
    )

    lines = []

    for block_index, block in enumerate(blocks, start=1):
        x0, y0, x1, y1, text, *_ = block

        lines.extend([
            (
                f"### Source Block {block_index:02d} "
                f"[bbox={x0:.1f},{y0:.1f},{x1:.1f},{y1:.1f}]"
            ),
            "",
            str(text).strip(),
            ""
        ])

    return "\n".join(lines).rstrip()


page_sections = []
conversion_rows = []
converted_page_text = {}

for page_index in range(PAGE_COUNT):

    page_number = page_index + 1
    page = pdf_document[page_index]
    native_text = page_native_text[page_number]

    pymupdf4llm_error = None

    try:
        primary_markdown = pymupdf4llm.to_markdown(
            str(SOURCE_PATH),
            pages=[page_index],
            write_images=False,
            embed_images=False,
            table_strategy="lines_strict",
            show_progress=False
        )

        primary_markdown = str(primary_markdown or "").strip()

    except Exception as error:
        primary_markdown = ""
        pymupdf4llm_error = str(error)

    primary_coverage = native_text_coverage(
        primary_markdown,
        native_text
    )

    if (
        primary_markdown
        and primary_coverage >= MIN_NATIVE_TOKEN_COVERAGE
    ):
        selected_text = primary_markdown
        selected_method = "pymupdf4llm"
        fallback_used = False

    else:
        selected_text = layout_block_fallback(page)
        selected_method = "PyMuPDF layout-aware native-block fallback"
        fallback_used = True

    selected_coverage = native_text_coverage(
        selected_text,
        native_text
    )

    converted_page_text[page_number] = selected_text

    page_sections.extend([
        f"## Source Page {page_number}",
        "",
        selected_text,
        ""
    ])

    conversion_rows.append({
        "Page Number": page_number,
        "Primary Method": "pymupdf4llm",
        "Primary Character Count": len(primary_markdown),
        "Native Token Coverage by Primary": primary_coverage,
        "Fallback Used": fallback_used,
        "Selected Method": selected_method,
        "Selected Character Count": len(selected_text),
        "Selected Native Token Coverage": selected_coverage,
        "pymupdf4llm Error": pymupdf4llm_error
    })


STRUCTURAL_MARKDOWN_TEXT = (
    "# D11 — Investor Presentation | Q4 & FY24\n\n"
    "> Complete Branch B structural representation of all ten physical PDF pages.\n"
    "> Page boundaries are explicit. OCR and semantic rewriting are not applied.\n\n"
    + "\n".join(page_sections).rstrip()
    + "\n"
)

STRUCTURAL_MARKDOWN_PATH.write_text(
    STRUCTURAL_MARKDOWN_TEXT,
    encoding="utf-8"
)

STRUCTURAL_MARKDOWN_SHA256 = sha256_file(
    STRUCTURAL_MARKDOWN_PATH
)

conversion_pages_df = pd.DataFrame(conversion_rows)

conversion_pages_df.to_csv(
    CONVERSION_PAGE_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Structural Markdown:", STRUCTURAL_MARKDOWN_PATH)
print("Representation SHA-256:", STRUCTURAL_MARKDOWN_SHA256)

display(conversion_pages_df)


In [ ]:
# ============================================================
# 3. Conversion-integrity verification
# ============================================================


# ------------------------------------------------------------
# 3.1 Page-boundary checks
# ------------------------------------------------------------

page_boundary_checks = {
    str(page_number):
        f"## Source Page {page_number}"
        in STRUCTURAL_MARKDOWN_TEXT

    for page_number
    in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
}


# ------------------------------------------------------------
# 3.2 Non-empty converted-page checks
# ------------------------------------------------------------

page_nonempty_checks = {
    str(page_number):
        bool(
            converted_page_text[
                page_number
            ].strip()
        )

    for page_number
    in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
}


# ------------------------------------------------------------
# 3.3 Native-token coverage checks
# ------------------------------------------------------------

selected_coverage_by_page = {
    str(
        int(
            row["Page Number"]
        )
    ):
        float(
            row[
                "Selected Native Token Coverage"
            ]
        )

    for _, row
    in conversion_pages_df.iterrows()
}


coverage_checks = {
    page_number:
        coverage
        >= MIN_NATIVE_TOKEN_COVERAGE

    for page_number, coverage
    in selected_coverage_by_page.items()
}


# ------------------------------------------------------------
# 3.4 Major document-region checks
# ------------------------------------------------------------

CONVERSION_MARKER_PATTERNS = {

    "presentation_title":
        (
            r"Investor Presentation\s*"
            r"\|\s*Q4\s*&\s*FY24"
        ),

    "safe_harbor":
        r"Safe Harbor",

    "management_commentary":
        r"Management Commentary",

    "quarterly_business_performance":
        r"Quarterly Business Performance",

    "profit_and_loss_statement":
        r"Profit\s*&\s*Loss Statement",

    "company_profile":
        (
            r"Our Legacy,\s*Our Future"
            r"|From 1978 till today"
        ),

    "corporate_timeline":
        (
            r"We Improve\.\s*Grow\.\s*Accelerate"
            r"|Established in 1978"
        ),

    "operational_footprint":
        (
            r"We serve multiple end markets"
            r"|Distributors spread across pin codes"
        )
}


conversion_marker_status = {
    marker:
        bool(
            re.search(
                pattern,
                STRUCTURAL_MARKDOWN_TEXT,
                flags=(
                    re.IGNORECASE
                    | re.DOTALL
                )
            )
        )

    for marker, pattern
    in CONVERSION_MARKER_PATTERNS.items()
}


# ------------------------------------------------------------
# 3.5 Diagnostic-text normalisation
# ------------------------------------------------------------

def normalise_integrity_text(text):

    text = str(
        text
    ).casefold()

    for character in [
        "*",
        "_",
        "`"
    ]:
        text = text.replace(
            character,
            ""
        )

    text = (
        text
        .replace("–", "-")
        .replace("—", "-")
    )

    text = " ".join(
        text.split()
    )

    return text


normalised_pages = {
    page_number:
        normalise_integrity_text(
            converted_page_text[
                page_number
            ]
        )

    for page_number
    in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
}


# ------------------------------------------------------------
# 3.6 Helper for order-independent page-specific checks
# ------------------------------------------------------------

def all_markers_present(
    page_number,
    markers
):
    """
    Verify that all source-derived markers survive within the
    specified converted physical page.

    Marker order is deliberately ignored because visual slides,
    charts and infographics may linearise differently during
    structural conversion.
    """

    page_text = normalised_pages[
        page_number
    ]

    return {
        marker:
            (
                normalise_integrity_text(
                    marker
                )
                in page_text
            )

        for marker
        in markers
    }


# ------------------------------------------------------------
# 3.7 Page 5 — Quarterly Business Performance
# ------------------------------------------------------------
#
# Page 5 is a three-panel stacked-bar chart.


PAGE_5_MARKERS = [
    "Quarterly Business Performance",
    "Net Revenue",
    "EBITDA",
    "Net Profit After Tax",
    "19,031",
    "22,293",
    "20,872",
    "3,343",
    "3,689",
    "2,849",
    "2,125",
    "2,518",
    "1,847",
    "FY22",
    "FY23",
    "FY24",
    "Q1",
    "Q2",
    "Q3",
    "Q4"
]


page_5_marker_checks = (
    all_markers_present(
        5,
        PAGE_5_MARKERS
    )
)


page_5_financial_chart_preserved = all(
    page_5_marker_checks.values()
)


# ------------------------------------------------------------
# 3.8 Page 6 — Profit & Loss table
# ------------------------------------------------------------

PAGE_6_MARKERS = [
    "Revenue from Operations",
    "EBITDA",
    "Profit After Tax",
    "EPS",
    "Q4 FY24",
    "Q4 FY23",
    "Q3 FY24",
    "FY24",
    "FY23"
]


page_6_marker_checks = (
    all_markers_present(
        6,
        PAGE_6_MARKERS
    )
)


page_6_pnl_preserved = all(
    page_6_marker_checks.values()
)


# ------------------------------------------------------------
# 3.9 Page 8 — Company profile
# ------------------------------------------------------------

PAGE_8_MARKERS = [
    "Our Legacy, Our Future",
    "Tarapur",
    "Daman",
    "Amravati",
    "Silvassa"
]


page_8_marker_checks = (
    all_markers_present(
        8,
        PAGE_8_MARKERS
    )
)


page_8_profile_preserved = all(
    page_8_marker_checks.values()
)


# ------------------------------------------------------------
# 3.10 Page 9 — Corporate timeline
# ------------------------------------------------------------

PAGE_9_MARKERS = [
    "We Improve. Grow. Accelerate",
    "1978-1987",
    "1991-2009",
    "2013-2020",
    "2021-2023",
    "Oxemberg",
    "J. Hampstead",
    "Cadini",
    "Ethnair"
]


page_9_marker_checks = (
    all_markers_present(
        9,
        PAGE_9_MARKERS
    )
)


page_9_timeline_preserved = all(
    page_9_marker_checks.values()
)


# ------------------------------------------------------------
# 3.11 Page 10 — Operational footprint
# ------------------------------------------------------------
#
# Page 10 is an infographic


PAGE_10_MARKERS = [
    "We serve multiple end markets",
    "800+",
    "~100",
    "245+",
    "~1.85",
    "~4.5",
    "5Mn and counting"
]


page_10_marker_checks = (
    all_markers_present(
        10,
        PAGE_10_MARKERS
    )
)


page_10_qualified_metrics_preserved = all(
    page_10_marker_checks.values()
)


# ------------------------------------------------------------
# 3.12 Combined representative-content status
# ------------------------------------------------------------

representative_content_status = {

    "page_5_financial_chart":
        page_5_financial_chart_preserved,

    "page_6_pnl_rows":
        page_6_pnl_preserved,

    "page_8_profile":
        page_8_profile_preserved,

    "page_9_timeline":
        page_9_timeline_preserved,

    "page_10_qualified_metrics":
        page_10_qualified_metrics_preserved
}


representative_content_preserved = all(
    representative_content_status.values()
)


# ------------------------------------------------------------
# 3.13 Fallback diagnostics
# ------------------------------------------------------------

fallback_page_count = int(
    conversion_pages_df[
        "Fallback Used"
    ].sum()
)


# ------------------------------------------------------------
# 3.14 Conversion-integrity object
# ------------------------------------------------------------

CONVERSION_INTEGRITY = {

    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "conversion_method":
        CONVERSION_METHOD,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "observed_page_count":
        PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "ocr_applied":
        False,

    # --------------------------------------------------------
    # Page retention
    # --------------------------------------------------------

    "all_source_pages_retained":
        all(
            page_boundary_checks.values()
        ),

    "page_boundary_checks":
        page_boundary_checks,

    "page_nonempty_checks":
        page_nonempty_checks,

    "all_converted_pages_nonempty":
        all(
            page_nonempty_checks.values()
        ),

    # --------------------------------------------------------
    # Source coverage
    # --------------------------------------------------------

    "minimum_native_token_coverage":
        MIN_NATIVE_TOKEN_COVERAGE,

    "selected_native_token_coverage_by_page":
        selected_coverage_by_page,

    "coverage_checks":
        coverage_checks,

    "all_pages_meet_source_coverage_threshold":
        all(
            coverage_checks.values()
        ),

    # --------------------------------------------------------
    # Major source regions
    # --------------------------------------------------------

    "conversion_marker_status":
        conversion_marker_status,

    "all_major_source_regions_preserved":
        all(
            conversion_marker_status.values()
        ),

    # --------------------------------------------------------
    # Representative page-specific integrity checks
    # --------------------------------------------------------

    "representative_content_status":
        representative_content_status,

    "representative_content_preserved":
        representative_content_preserved,

    "page_5_marker_checks":
        page_5_marker_checks,

    "page_6_marker_checks":
        page_6_marker_checks,

    "page_8_marker_checks":
        page_8_marker_checks,

    "page_9_marker_checks":
        page_9_marker_checks,

    "page_10_marker_checks":
        page_10_marker_checks,

    # --------------------------------------------------------
    # Conversion implementation
    # --------------------------------------------------------

    "fallback_page_count":
        fallback_page_count,

    # --------------------------------------------------------
    # Explicitly excluded operations
    # --------------------------------------------------------

    "complete_source_document_retained":
        True,

    "scope_filtering_applied":
        False,

    "page_removal_applied":
        False,

    "page_cropping_applied":
        False,

    "native_text_duplication_applied":
        False,

    "chart_value_reconstruction_applied":
        False,

    "table_value_reconstruction_applied":
        False,

    "timeline_reconstruction_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "normalisation_applied":
        False,

    # --------------------------------------------------------
    # Overall result
    # --------------------------------------------------------

    "conversion_integrity_passed":
        all([
            SOURCE_HASH_MATCH,
            PAGE_COUNT_VALID,
            TEXT_EXTRACTABLE,
            not OCR_REQUIRED,

            all(
                page_boundary_checks.values()
            ),

            all(
                page_nonempty_checks.values()
            ),

            all(
                coverage_checks.values()
            ),

            all(
                conversion_marker_status.values()
            ),

            representative_content_preserved
        ])
}


# ------------------------------------------------------------
# 3.15 Save conversion-integrity report
# ------------------------------------------------------------

CONVERSION_INTEGRITY_PATH.write_text(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 3.16 Display result
# ------------------------------------------------------------

print(
    json.dumps(
        CONVERSION_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)


# ------------------------------------------------------------
# 3.17 Stop if integrity genuinely failed
# ------------------------------------------------------------

if not CONVERSION_INTEGRITY[
    "conversion_integrity_passed"
]:
    raise ValueError(
        "D11 Branch B structural conversion "
        "failed integrity checks."
    )

In [ ]:
# ============================================================
# 4. Preservation of representation metadata
# ============================================================

REPRESENTATION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Complete page-aware structural Markdown",
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": EXPECTED_PAGE_COUNT,
    "branch_name":
        BRANCH_NAME,
    "conversion_method":
        CONVERSION_METHOD,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,
    "structural_conversion_applied": True,
    "primary_converter": "pymupdf4llm",
    "fallback_method":
        "PyMuPDF layout-aware native text blocks with bounding boxes",
    "fallback_trigger":
        f"native token coverage < {MIN_NATIVE_TOKEN_COVERAGE}",
    "fallback_page_count": fallback_page_count,
    "complete_source_document_retained": True,
    "page_boundaries_made_explicit": True,
    "scope_enforced_by_prompt_not_representation_filtering": True,
    "divider_pages_retained_in_representation": True,
    "native_text_duplication_applied": False,
    "ocr_applied": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "chart_value_reconstruction_applied": False,
    "table_value_reconstruction_applied": False,
    "timeline_reconstruction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_PATH.write_text(
    json.dumps(REPRESENTATION, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(REPRESENTATION, ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# 5. Creation of controlled Branch B prompt
# ============================================================

BRANCH_B_PROMPT = r"""You are an information extraction assistant.

Extract the predefined financial, corporate-profile, historical and
operational records represented within the defined scope of the
attached original PDF investor presentation:

Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24.

Treat the attached original PDF as the only source of information.

For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Use exactly one of these Category values:

- Presentation metadata
- Management commentary
- Quarterly business performance
- Profit and loss statement
- Company profile
- Corporate timeline
- Operational footprint


1. Presentation metadata

From physical PDF pages 1 and 2, extract the predefined metadata
concepts concerning:

- the presentation title;
- the company name;
- the Safe Harbor status/purpose statement.

Represent the Safe Harbor material as one principal metadata record.
Do not extract the complete legal disclaimer sentence-by-sentence.


2. Management commentary

From physical PDF page 4, extract the explicitly represented
management-commentary observations concerning:

- market conditions;
- Revenue from Operations and its comparative period;
- the revenue mix by Fabric, Garments, and Yarn & Others;
- EBITDA and EBITDA Margin;
- Profit After Tax and PAT Margin;
- retail footprint;
- current and comparative sales-promotion spending;
- the approved dividend;
- the dividend percentage;
- the share face value associated with the dividend statement;
- the identified management spokesperson.

Extract the values and periods directly from the page.

Keep independently represented observations separate even when a
similar metric occurs elsewhere in the presentation.

Do not use values from another page to complete this source section.


3. Quarterly Business Performance

From the chart on physical PDF page 5, extract every explicitly printed
annual total and every explicitly printed quarterly component for:

- Net Revenue;
- EBITDA;
- Net Profit After Tax.

For each metric:

- preserve the annual totals for each represented fiscal year;
- preserve every explicitly labelled Q1, Q2, Q3 and Q4 component;
- associate each quarterly value with the correct fiscal year;
- preserve the represented monetary scale;
- preserve the chart's stated qualification concerning standalone
  financials and rounding where relevant in Description.

Extract only values numerically printed in the chart.

Do not estimate values from bar height, bar area, graphical position,
colour, or proportional size.

Annual totals and quarterly components are separate records.

Do not merge chart observations with similar values represented in
management commentary or the Profit & Loss Statement.


4. Q4FY24 Profit & Loss Statement

From the table on physical PDF page 6, extract every explicitly
populated numerical observation represented in the body of the table.

Use the financial row label as Topic.

For the main period columns, preserve observations under:

- Q4 FY24;
- Q4 FY23;
- Q3 FY24;
- FY24;
- FY23.

For populated YoY and QoQ cells:

- create separate records;
- distinguish Year-on-Year from Quarter-on-Quarter change in Topic;
- preserve the correct comparison in Reporting Period;
- preserve percentages as percentages.

Do not create records for blank YoY or QoQ cells.

Treat margin rows as percentages.

Treat EPS using its represented rupee-per-share scale rather than
the table-level Rs. Mn scale.

Also extract the two explicitly represented marketing and sales
promotion expense observations in the page-6 footnote.

Do not calculate any missing YoY, QoQ, margin or other value.

Do not recompute or reconcile totals.

Do not merge page-6 observations with similar values printed on
other pages.


5. Company profile

From physical PDF page 8, extract the principal represented
company-profile observations concerning:

- company history;
- market position;
- product categories;
- explicitly listed brands and sub-brands;
- retail and online presence;
- manufacturing certifications;
- manufacturing locations;
- distribution ecosystem.

Preserve explicitly represented names, locations and certification
wording.

Do not create records from decorative imagery or the closing tagline.


6. Corporate timeline

From physical PDF page 9, extract every explicitly listed milestone
within the four represented timeline phases.

Preserve:

- the milestone subject as Topic;
- the represented milestone wording in Description or Value;
- the phase period as Reporting Period.

Do not infer exact event years where only a phase-level period is
represented.

Do not create records from decorative images or phase numbering alone.


7. Operational footprint

From physical PDF page 10, extract every prominently represented
operational metric concerning:

- distributors;
- fabric sold;
- stores across the nation;
- retail space;
- apparels sold;
- customers served.

Also extract one record representing the explicitly listed commercial
channels/end markets.

Preserve approximation, lower-bound and continuation wording exactly
when it forms part of a represented value.

For example, if a printed value contains an approximation symbol,
a plus sign or wording such as “and counting”, preserve that
qualification rather than silently converting it into an exact number.

Do not replace a page-10 observation with a similar value represented
elsewhere in the document.


Excluded source regions

Physical PDF pages 3 and 7 are section-divider slides and do not
contribute extraction records within this task.

Decorative imagery and logos are outside the extraction scope.


Field rules:

Category:
- Use exactly one of the seven Category labels defined above.

Topic:
- Use a concise stable label describing the represented metric,
  statement, milestone or concept.
- Preserve source terminology for financial metrics.

Description:
- Provide a concise source-grounded description of the observation.
- Preserve material source qualifications where relevant.
- Do not introduce external interpretation.

Value:
- Use a JSON number when the source represents an unqualified numeric
  value.
- Use a JSON string when the value is textual or when qualification
  such as "~", "+", or "and counting" is semantically part of the
  represented value.
- Use null only where no separate Value applies.
- Preserve negative signs.
- Do not calculate, estimate, derive, convert or correct values.

Unit:
- Preserve the represented measurement scale.
- Use consistent source-grounded forms such as:
  text
  Rs. Mn
  Rs. crores
  percent
  stores
  distributors
  Mn meters
  L sqft
  Mn pieces
  Mn customers
  Rs. per share
  Rs.
  year
- Do not silently rescale monetary values.

Reporting Period:
- Derive the period directly from the represented source.
- Preserve fiscal-year and quarter distinctions.
- Preserve comparison periods for YoY and QoQ records.
- Use the represented timeline phase period for timeline milestones.
- Do not infer an exact year where only a phase period is represented.

Source Location:
- Use the physical PDF page containing the observation.
- Use the form:
  "PDF page N"

Additional rules:

- Use only information explicitly represented in the supplied structural Markdown representation.
- Preserve repeated observations when they occur independently in
  different source sections.
- Do not deduplicate records solely because Topic or Value is repeated.
- Do not estimate chart values visually.
- Do not extract blank table cells.
- Preserve negative percentages.
- Preserve approximation and lower-bound wording.
- Do not use external knowledge.
- Do not follow external links.
- Do not calculate missing values.
- Do not normalise or convert measurement scales.
- Do not silently correct source wording.
- Do not extract records outside the predefined source scope.
- Verify that every item within the defined scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D11",
  "branch": "B",
  "records": [
    {
      "Category": null,
      "Topic": null,
      "Description": null,
      "Value": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object."""

PROMPT_PATH.write_text(
    BRANCH_B_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print("Prompt saved:", PROMPT_PATH.name)
print("Prompt SHA-256:", PROMPT_SHA256)


In [ ]:
# ============================================================
# 6. Pre-extraction experiment metadata
# ============================================================

EXPERIMENT_METADATA_PRE = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_FORMAT,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "complete_source_document_retained": True,
    "scope_filtering_applied": False,
    "page_boundaries_made_explicit": True,
    "divider_pages_retained_in_representation": True,
    "fallback_page_count": fallback_page_count,
    "native_text_duplication_applied": False,
    "ocr_applied_for_model_input": False,
    "normalisation_applied": False,
    "semantic_rewriting_applied": False,
    "unit_conversion_applied": False,
    "numeric_calculation_applied": False,
    "chart_value_reconstruction_applied": False,
    "table_value_reconstruction_applied": False,
    "timeline_reconstruction_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "expected_extraction_scope": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
        "expected_fields": EXPECTED_FIELDS
    },
    "reference_expectations_disclosed_to_model": False,
    "content_validation_performed":
        False,
    "source_diagnostics_file": SOURCE_DIAGNOSTICS_PATH.name,
    "page_diagnostics_file": PAGE_DIAGNOSTICS_PATH.name,
    "conversion_page_audit_file": CONVERSION_PAGE_AUDIT_PATH.name,
    "conversion_integrity_file": CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format":
        "JSON object with document_id, branch and records",
    "execution_environment": "Independent ChatGPT conversation",
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

page_diagnostics_df.to_csv(
    PAGE_DIAGNOSTICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(json.dumps(EXPERIMENT_METADATA_PRE, ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# 7. Download model-input artefacts
# ============================================================

for path in [
    SOURCE_DIAGNOSTICS_PATH,
    PAGE_DIAGNOSTICS_PATH,
    CONVERSION_PAGE_AUDIT_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D11_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D11_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original PDF, Stage 1 reference dataset, "
    "Branch A outputs, Validation A outputs, or expected counts.\n"
    "5. Save the first complete model response exactly as returned as TXT.\n"
    "6. Do not correct, repair, reorder, deduplicate, or regenerate it."
)


In [ ]:
# ============================================================
# 8. Upload and preservation of untouched model response
# ============================================================

uploaded_output = files.upload()

txt_paths = [
    Path(name)
    for name in uploaded_output
    if name.lower().endswith(".txt")
]

if len(txt_paths) != 1:
    raise ValueError(
        "Upload exactly one TXT file containing the complete D11 Branch B response."
    )

UPLOADED_RAW_RESPONSE_PATH = txt_paths[0]

RAW_RESPONSE_TEXT = UPLOADED_RAW_RESPONSE_PATH.read_text(
    encoding="utf-8"
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError("The uploaded D11 Branch B response is empty.")

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw response preserved:", RAW_RESPONSE_PATH.name)
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 9. Parsing without repairing the response
# ============================================================

valid_json = False
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(RAW_RESPONSE_TEXT)
    valid_json = True
except json.JSONDecodeError as error:
    json_parsing_error = str(error)

top_level_object_valid = (
    valid_json and isinstance(parsed_response, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_response
)

records_is_list = (
    records_present
    and isinstance(parsed_response.get("records"), list)
)

records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)


print("Valid JSON:", valid_json)
print("JSON parsing error:", json_parsing_error)
print("Document ID correct:", document_id_correct)
print("Branch correct:", branch_correct)
print("Records evaluable:", records_evaluable)
print("Observed record count:", observed_record_count)


In [ ]:
# ============================================================
# 10. Schema, field-type and D11 scope diagnostics
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

if records_evaluable:

    for record_index, record in enumerate(extracted_records):

        if not isinstance(record, dict):
            record_structure_issues.append({
                "record_index": record_index,
                "issue": "Record is not a JSON object"
            })
            continue

        observed_fields = list(record.keys())

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index": record_index,
                "issue": "Field names or field order differ",
                "expected_fields": EXPECTED_FIELDS,
                "observed_fields": observed_fields
            })

        for field in STRING_OR_NULL_FIELDS:

            value = record.get(field)

            if (
                value is not None
                and not isinstance(value, str)
            ):
                field_type_issues.append({
                    "record_index": record_index,
                    "field": field,
                    "observed_type": type(value).__name__,
                    "expected_type": "string or null"
                })

        value = record.get("Value")

        if (
            isinstance(value, bool)
            or not isinstance(value, VALUE_ALLOWED_TYPES)
        ):
            field_type_issues.append({
                "record_index": record_index,
                "field": "Value",
                "observed_type": type(value).__name__,
                "expected_type": "string, number or null"
            })

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(field)

            if (
                value is None
                or (
                    isinstance(value, str)
                    and not value.strip()
                )
            ):
                missing_mandatory_values.append({
                    "record_index": record_index,
                    "field": field
                })


record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)


if records_evaluable:

    record_count_valid = (
        observed_record_count == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )

    category_counts_valid = (
        observed_category_counts == EXPECTED_CATEGORY_COUNTS
    )

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field in EXPECTED_FIELDS
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    duplicate_record_count = sum(
        1
        for count in duplicate_counter.values()
        if count > 1
    )

    def source_page(record):
        value = record.get("Source Location")
        if not isinstance(value, str):
            return None

        match = re.fullmatch(
            r"PDF page\s+(\d+)",
            value.strip(),
            flags=re.IGNORECASE
        )

        return int(match.group(1)) if match else None

    observed_source_pages = [
        source_page(record)
        for record in extracted_records
        if isinstance(record, dict)
    ]

    physical_page_references_valid = all(
        page is not None
        and 1 <= page <= EXPECTED_PAGE_COUNT
        for page in observed_source_pages
    )

    divider_pages_excluded = not any(
        page in EXCLUDED_DIVIDER_PAGES
        for page in observed_source_pages
        if page is not None
    )

    qualified_values_observed = {
        str(record.get("Value"))
        for record in extracted_records
        if isinstance(record, dict)
        and str(record.get("Value")) in EXPECTED_QUALIFIED_VALUES
    }

    qualified_values_preserved = (
        qualified_values_observed == EXPECTED_QUALIFIED_VALUES
    )

    negative_numeric_or_text_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and (
                (
                    isinstance(record.get("Value"), (int, float))
                    and not isinstance(record.get("Value"), bool)
                    and record.get("Value") < 0
                )
                or (
                    isinstance(record.get("Value"), str)
                    and record.get("Value").strip().startswith("-")
                )
            )
        )
    )

    page_5_record_count = sum(
        1 for page in observed_source_pages
        if page == 5
    )

    page_6_record_count = sum(
        1 for page in observed_source_pages
        if page == 6
    )

else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_record_count = None
    observed_source_pages = None
    physical_page_references_valid = None
    divider_pages_excluded = None
    qualified_values_observed = None
    qualified_values_preserved = None
    negative_numeric_or_text_count = None
    page_5_record_count = None
    page_6_record_count = None


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        observed_record_count,
    "record_count_matches_reference":
        record_count_valid,
    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "categories_valid":
        categories_valid,
    "category_counts_match_reference":
        category_counts_valid,
    "mandatory_fields_complete":
        mandatory_fields_complete,
    "missing_mandatory_value_count":
        (
            len(missing_mandatory_values)
            if records_evaluable
            else None
        ),
    "duplicate_complete_record_signature_count":
        duplicate_record_count,
    "physical_page_references_valid":
        physical_page_references_valid,
    "divider_pages_excluded":
        divider_pages_excluded,
    "expected_qualified_values":
        sorted(EXPECTED_QUALIFIED_VALUES),
    "qualified_values_observed":
        (
            sorted(qualified_values_observed)
            if qualified_values_observed is not None
            else None
        ),
    "qualified_values_preserved":
        qualified_values_preserved,
    "negative_numeric_or_text_value_count":
        negative_numeric_or_text_count,
    "page_5_record_count":
        page_5_record_count,
    "page_6_record_count":
        page_6_record_count
}

print(json.dumps(CONTENT_DIAGNOSTICS, ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# 11. Determine technical/schema validity
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues
        if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        field_type_issues
        if records_evaluable else None,

    "missing_mandatory_values":
        missing_mandatory_values
        if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "scope_complete":
        bool(record_count_valid)
        if record_count_valid is not None
        else False,

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
parsed_extraction_created = False
parsed_extraction_sha256 = None

if structurally_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get("document_id"),

        "branch":
            parsed_response.get("branch"),

        "records":
            extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_created = True

    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:
    print(
        "No parsed extraction created because "
        "the output is not structurally evaluable."
    )

In [ ]:
# ============================================================
# 12. Final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,
    "raw_response_file": RAW_RESPONSE_PATH.name,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),
    "parsed_extraction_sha256":
        parsed_extraction_sha256,
    "json_valid": valid_json,
    "records_evaluable": records_evaluable,
    "observed_record_count": observed_record_count,
    "observed_category_counts": observed_category_counts,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "content_validation_performed":
        False,
    "notes": (
        "D11 Branch B converts the complete ten-page investor-presentation "
        "PDF to page-aware structural Markdown. pymupdf4llm is the primary "
        "converter; pages below the fixed native-token coverage threshold "
        "use a non-duplicating layout-aware native-block fallback. "
        "No OCR, semantic rewriting, normalisation, unit conversion, "
        "numeric calculation, chart/table value reconstruction, or manual "
        "correction is applied. Stage 1 expected counts are used only after "
        "extraction for diagnostics. Content-level validation is performed "
        "separately in Validation B — D11."
    )
}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "llm_input_representation":
        LLM_INPUT_REPRESENTATION,

    "conversion_method":
        CONVERSION_METHOD,
    "representation_file": STRUCTURAL_MARKDOWN_PATH.name,
    "representation_sha256": STRUCTURAL_MARKDOWN_SHA256,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "complete_source_document_retained": True,
    "fallback_page_count": fallback_page_count,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches": record_count_valid,
    "expected_category_counts": EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts": observed_category_counts,
    "category_counts_match": category_counts_valid,
    "valid_json": valid_json,
    "records_evaluable": records_evaluable,
    "record_schema_valid": record_schema_valid,
    "field_types_valid": field_types_valid,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),
    "scope_complete": record_count_valid,
    "duplicate_complete_record_signature_count":
        duplicate_record_count,
    "physical_page_references_valid":
        physical_page_references_valid,
    "divider_pages_excluded":
        divider_pages_excluded,
    "qualified_values_preserved":
        qualified_values_preserved,
    "page_5_record_count":
        page_5_record_count,
    "page_6_record_count":
        page_6_record_count,
    "parsed_extraction_created":
        bool(structurally_evaluable),
    "content_validation_performed":
        False,

    "notes": (
        "Content-level validation is performed separately "
        "in Validation B — D11."
    )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_SUMMARY, ensure_ascii=False, indent=2))


In [ ]:
# ============================================================
# 13. Final artefact inventory and downloads
# ============================================================

GENERATED_OUTPUTS = [
    SOURCE_DIAGNOSTICS_PATH,
    PAGE_DIAGNOSTICS_PATH,
    CONVERSION_PAGE_AUDIT_PATH,
    STRUCTURAL_MARKDOWN_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    GENERATED_OUTPUTS.append(PARSED_EXTRACTION_PATH)

print("Generated D11 Branch B files:")

for path in GENERATED_OUTPUTS:
    print("-", path.name, "| exists:", path.exists())

for path in GENERATED_OUTPUTS:
    files.download(path)